# Notebook 2: Baseline Models & Dataset Classes

This notebook implements:
1. Data loading and preprocessing functions
2. Dataset classes for PyTorch
3. All 6 baseline model architectures:
   - MLP Baseline
   - 2D-CNN Single Frame
   - 2D-CNN Multi-Frame
   - EEGNet
   - DE-CNN (VIGNet-style)
   - Plain Channel Transformer

In [1]:
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080
CUDA Version: 12.8


## 1. Data Loading Functions

In [2]:
def load_seedvig_5band_all_features(base_dir, feature_key='de_LDS'):
    """
    Load one feature type from all sessions (23 total).
    Each session is treated as a separate 'subject' for splitting.
    
    Args:
        base_dir: Path to SEED-VIG directory
        feature_key: 'de_LDS' | 'de_movingAve' | 'psd_LDS' | 'psd_movingAve'
    
    Returns:
        X_all: (N, 17, 5) - all EEG features
        y_all: (N,) - all PERCLOS labels [0,1]
        subj_all: (N,) - session ID for each sample (1-23)
    """
    base_dir = Path(base_dir)
    eeg_dir = base_dir / 'EEG_Feature_5Bands'
    label_dir = base_dir / 'perclos_labels'
    
    eeg_files = sorted(list(eeg_dir.glob('*.mat')))
    label_files = sorted(list(label_dir.glob('*.mat')))
    
    X_list, y_list, subj_list = [], [], []
    
    # Treat each session as a separate subject (session_id from 1 to 23)
    for session_id, (eeg_file, label_file) in enumerate(zip(eeg_files, label_files), start=1):
        # Load data
        eeg_data = sio.loadmat(str(eeg_file))
        label_data = sio.loadmat(str(label_file))
        
        # Extract features (17, 885, 5) -> transpose to (885, 17, 5)
        X = eeg_data[feature_key].transpose(1, 0, 2)
        
        # Extract PERCLOS labels
        perclos_key = 'perclos' if 'perclos' in label_data else list(label_data.keys())[0]
        y = label_data[perclos_key].squeeze()
        
        X_list.append(X)
        y_list.append(y)
        # Use session_id instead of subject_id
        subj_list.append(np.full(len(y), session_id))
    
    X_all = np.concatenate(X_list, axis=0).astype(np.float32)
    y_all = np.concatenate(y_list, axis=0).astype(np.float32)
    subj_all = np.concatenate(subj_list, axis=0).astype(np.int32)
    
    print(f"Loaded {len(eeg_files)} sessions (each treated as separate subject)")
    print(f"  X shape: {X_all.shape}")
    print(f"  y shape: {y_all.shape}")
    print(f"  Unique sessions: {len(np.unique(subj_all))}")
    
    return X_all, y_all, subj_all

## 2. Normalization Function

In [3]:
def normalize_features_subjectwise(X_all, subj_all):
    """
    Z-score normalize per subject: mean=0, std=1.
    
    Args:
        X_all: (N, 17, 5)
        subj_all: (N,)
    
    Returns:
        X_norm: (N, 17, 5) normalized
    """
    X_norm = np.zeros_like(X_all)
    
    for subj_id in np.unique(subj_all):
        idx = np.where(subj_all == subj_id)[0]
        X_s = X_all[idx]  # (Ns, 17, 5)
        
        # Flatten to (Ns, 85)
        X_flat = X_s.reshape(X_s.shape[0], -1)
        
        # Z-score normalize
        mean = X_flat.mean(axis=0, keepdims=True)
        std = X_flat.std(axis=0, keepdims=True) + 1e-8
        X_norm_flat = (X_flat - mean) / std
        
        # Reshape back
        X_norm[idx] = X_norm_flat.reshape(-1, 17, 5)
    
    print(f"Normalization complete (subject-wise z-score)")
    return X_norm

## 3. Train/Test Split Function

In [4]:
def subject_wise_split(subj_all, train_ratio=0.80, val_ratio=0.10, seed=42):
    """
    Split subjects into train, validation, and test sets.
    
    Args:
        subj_all: (N,) subject IDs
        train_ratio: fraction for training (default 0.80)
        val_ratio: fraction for validation (default 0.10)
        seed: random seed
    
    Returns:
        train_mask: boolean mask for training samples
        val_mask: boolean mask for validation samples
        test_mask: boolean mask for test samples
    """
    rng = np.random.default_rng(seed)
    unique_subj = np.unique(subj_all)
    rng.shuffle(unique_subj)
    
    # Calculate splits
    n_total = len(unique_subj)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    # test gets remaining subjects
    
    train_subj = unique_subj[:n_train]
    val_subj = unique_subj[n_train:n_train+n_val]
    test_subj = unique_subj[n_train+n_val:]
    
    train_mask = np.isin(subj_all, train_subj)
    val_mask = np.isin(subj_all, val_subj)
    test_mask = np.isin(subj_all, test_subj)
    
    print(f"Subject split (80-10-10):")
    print(f"  Train: {len(train_subj)} subjects ({train_subj.tolist()}), {train_mask.sum()} samples")
    print(f"  Val:   {len(val_subj)} subjects ({val_subj.tolist()}), {val_mask.sum()} samples")
    print(f"  Test:  {len(test_subj)} subjects ({test_subj.tolist()}), {test_mask.sum()} samples")
    
    return train_mask, val_mask, test_mask

## 4. Dataset Classes

In [5]:
class SEEDVIGSingleFrameDataset(Dataset):
    """
    Dataset for single frames (no temporal context).
    
    Args:
        X: (N, 17, 5)
        y: (N,) labels
        flatten: True for MLP (returns 85-dim), False for CNN (returns 17×5)
        regression: True for continuous labels, False for classification
    """
    def __init__(self, X, y, flatten=False, regression=False):
        self.X = torch.from_numpy(X).float()
        self.flatten = flatten
        self.regression = regression
        
        if regression:
            self.y = torch.from_numpy(y).float()
        else:
            self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]  # (17, 5)
        
        if self.flatten:
            x = x.reshape(-1)  # (85,) - use reshape instead of view for non-contiguous tensors
        
        return x, self.y[idx]


class SEEDVIGMultiFrameDataset(Dataset):
    """
    Dataset for temporal sequences within each subject.
    
    Args:
        X: (N, 17, 5)
        y: (N,) labels
        subj_ids: (N,) subject IDs
        T_seq: sequence length
        step: sliding window stride
        regression: True for continuous labels
    """
    def __init__(self, X, y, subj_ids, T_seq=5, step=2, regression=False):
        self.T_seq = T_seq
        self.regression = regression
        
        # Build sequences per subject
        self.sequences = []
        self.labels = []
        
        for subj_id in np.unique(subj_ids):
            idx = np.where(subj_ids == subj_id)[0]
            X_s = X[idx]
            y_s = y[idx]
            
            # Create sliding windows
            for i in range(0, len(X_s) - T_seq + 1, step):
                seq = X_s[i:i+T_seq]  # (T_seq, 17, 5)
                
                # Label: majority vote for classification, mean for regression
                if regression:
                    label = np.mean(y_s[i:i+T_seq])
                else:
                    label = np.round(np.mean(y_s[i:i+T_seq])).astype(int)
                
                self.sequences.append(seq)
                self.labels.append(label)
        
        self.sequences = torch.from_numpy(np.array(self.sequences)).float()
        
        if regression:
            self.labels = torch.from_numpy(np.array(self.labels)).float()
        else:
            self.labels = torch.from_numpy(np.array(self.labels)).long()
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

print("Dataset classes defined successfully!")

Dataset classes defined successfully!


## 5. Model 1: MLP Baseline

In [6]:
class MLPBaseline(nn.Module):
    """
    Simple 3-layer fully-connected network.
    Input: (batch, 85) flattened features
    Output: (batch, num_outputs)
    """
    def __init__(self, input_size=85, num_outputs=2, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, num_outputs)
        )
    
    def forward(self, x):
        # x: (B, 85)
        return self.network(x)

print("MLPBaseline defined!")

MLPBaseline defined!


## 6. Model 2: 2D-CNN Single Frame

In [7]:
class CNN2DSingleFrame(nn.Module):
    """
    Treats (17, 5) as a tiny 2D "image".
    Input: (batch, 17, 5)
    Output: (batch, num_outputs)
    """
    def __init__(self, num_outputs=2, dropout=0.3):
        super().__init__()
        
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1))  # Pool only channels: (17,5)→(8,5)
        )
        
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))  # Global pooling → (64,1,1)
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_outputs)
        )
    
    def forward(self, x):
        # x: (B, 17, 5)
        if x.dim() == 3:
            x = x.unsqueeze(1)  # Add channel: (B, 1, 17, 5)
        
        x = self.conv_block1(x)  # (B, 32, 8, 5)
        x = self.conv_block2(x)  # (B, 64, 1, 1)
        x = self.fc(x)           # (B, num_outputs)
        return x

print("CNN2DSingleFrame defined!")

CNN2DSingleFrame defined!


## 7. Model 3: 2D-CNN Multi-Frame

In [8]:
class CNN2DMultiFrame(nn.Module):
    """
    Processes multiple consecutive frames to capture temporal dynamics.
    Input: (batch, T_seq, 17, 5)
    Output: (batch, num_outputs)
    """
    def __init__(self, T_seq=5, num_outputs=2, dropout=0.3):
        super().__init__()
        self.T_seq = T_seq
        
        # Spatial CNN (processes individual frames)
        self.spatial_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        # Temporal CNN (over frame sequence)
        self.temporal_cnn = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2) if T_seq >= 4 else nn.Identity(),
            
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # Global temporal pooling
        )
        
        # Output head
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_outputs)
        )
    
    def forward(self, x):
        # x: (B, T, 17, 5)
        B, T, C, F = x.shape
        
        # Process each frame independently
        x_flat = x.view(B * T, C, F).unsqueeze(1)  # (B*T, 1, 17, 5)
        frame_features = self.spatial_cnn(x_flat)   # (B*T, 64, 1, 1)
        frame_features = frame_features.view(B, T, 64)  # (B, T, 64)
        
        # Temporal aggregation
        frame_features = frame_features.transpose(1, 2)  # (B, 64, T)
        temporal_features = self.temporal_cnn(frame_features)  # (B, 128, 1)
        
        # Classification/regression
        out = self.fc(temporal_features)  # (B, num_outputs)
        return out

print("CNN2DMultiFrame defined!")

CNN2DMultiFrame defined!


## 8. Model 4: EEGNet

In [9]:
class EEGNet(nn.Module):
    """
    Compact CNN specifically designed for EEG signals.
    Adapted from: Lawhern et al. (2018) "EEGNet: A Compact CNN for EEG-based BCIs"
    
    Input: (batch, 17, 5)
    Output: (batch, num_outputs)
    """
    def __init__(self, num_outputs=2, dropout=0.25, F1=8, D=2, F2=16):
        super().__init__()
        
        # Block 1: Temporal convolution
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(F1)
        )
        
        # Block 2: Depthwise spatial convolution
        self.conv2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(17, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 2)),
            nn.Dropout(dropout)
        )
        
        # Block 3: Separable convolution
        self.conv3 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 3), padding=(0, 1), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 2)),
            nn.Dropout(dropout)
        )
        
        # Classifier
        self.fc = nn.Linear(F2, num_outputs)
    
    def forward(self, x):
        # x: (B, 17, 5) -> reshape to (B, 1, 17, 5)
        if x.dim() == 3:
            x = x.unsqueeze(1)  # (B, 1, 17, 5)
        
        x = self.conv1(x)  # (B, 8, 17, 5)
        x = self.conv2(x)  # (B, 16, 1, 2)
        x = self.conv3(x)  # (B, 16, 1, 1)
        
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return x

print("EEGNet defined!")

EEGNet defined!


## 9. Model 5: DE-CNN (VIGNet-style)

In [10]:
class DECNN(nn.Module):
    """
    Deep CNN inspired by VIGNet for vigilance estimation.
    Processes differential entropy features hierarchically.
    
    Input: (batch, 17, 5)
    Output: (batch, num_outputs)
    """
    def __init__(self, num_outputs=2, dropout=0.4):
        super().__init__()
        
        # Spatial feature extraction
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),  # (32, 8, 5)
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),  # (64, 4, 5)
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))  # (128, 1, 1)
        )
        
        # Deep classification head
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(64, num_outputs)
        )
    
    def forward(self, x):
        # x: (B, 17, 5)
        if x.dim() == 3:
            x = x.unsqueeze(1)  # (B, 1, 17, 5)
        
        x = self.conv_layers(x)  # (B, 128, 1, 1)
        x = self.fc(x)           # (B, num_outputs)
        return x

print("DECNN defined!")

DECNN defined!


## 10. Model 6: Plain Channel Transformer

In [11]:
class ChannelTransformer(nn.Module):
    """
    Transformer-based model treating each channel as a token.
    Uses self-attention to model inter-channel relationships.
    
    Input: (batch, 17, 5)
    Output: (batch, num_outputs)
    """
    def __init__(self, num_outputs=2, embed_dim=64, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        
        # Channel embedding: (5 bands) -> (embed_dim)
        self.channel_embed = nn.Linear(5, embed_dim)
        
        # Positional encoding for 17 channels
        self.pos_encoding = nn.Parameter(torch.randn(1, 17, embed_dim))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=128,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_outputs)
        )
    
    def forward(self, x):
        # x: (B, 17, 5)
        B = x.size(0)
        
        # Embed each channel
        x = self.channel_embed(x)  # (B, 17, embed_dim)
        
        # Add positional encoding
        x = x + self.pos_encoding  # (B, 17, embed_dim)
        
        # Transformer encoding
        x = self.transformer(x)  # (B, 17, embed_dim)
        
        # Mean pooling over 17 tokens
        x = x.mean(dim=1)  # (B, embed_dim)
        
        # Classification
        x = self.fc(x)  # (B, num_outputs)
        return x

print("ChannelTransformer defined!")

ChannelTransformer defined!


## 11. Test All Models with Dummy Data

In [12]:
def test_model(model, input_shape, model_name):
    """
    Test a model with dummy data and count parameters.
    """
    dummy_input = torch.randn(4, *input_shape)  # Batch size 4
    
    # Forward pass
    try:
        output = model(dummy_input)
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"\n{model_name}:")
        print(f"  Input shape:  {dummy_input.shape}")
        print(f"  Output shape: {output.shape}")
        print(f"  Total params: {total_params:,}")
        print(f"  Trainable:    {trainable_params:,}")
        print(f"  ✓ Test passed!")
        
        return True
    except Exception as e:
        print(f"\n{model_name}:")
        print(f"  ✗ Error: {e}")
        return False

print("\n" + "="*60)
print("Testing All Models")
print("="*60)

# Test MLP
mlp = MLPBaseline(input_size=85, num_outputs=2)
test_model(mlp, (85,), "MLP Baseline")

# Test CNN Single
cnn_single = CNN2DSingleFrame(num_outputs=2)
test_model(cnn_single, (17, 5), "CNN2D Single Frame")

# Test CNN Multi
cnn_multi = CNN2DMultiFrame(T_seq=5, num_outputs=2)
test_model(cnn_multi, (5, 17, 5), "CNN2D Multi-Frame (T=5)")

# Test EEGNet
eegnet = EEGNet(num_outputs=2)
test_model(eegnet, (17, 5), "EEGNet")

# Test DE-CNN
decnn = DECNN(num_outputs=2)
test_model(decnn, (17, 5), "DE-CNN (VIGNet-style)")

# Test Transformer
transformer = ChannelTransformer(num_outputs=2)
test_model(transformer, (17, 5), "Channel Transformer")

print("\n" + "="*60)
print("All models tested successfully!")
print("="*60)


Testing All Models

MLP Baseline:
  Input shape:  torch.Size([4, 85])
  Output shape: torch.Size([4, 2])
  Total params: 21,410
  Trainable:    21,410
  ✓ Test passed!

CNN2D Single Frame:
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 23,298
  Trainable:    23,298
  ✓ Test passed!

CNN2D Multi-Frame (T=5):
  Input shape:  torch.Size([4, 5, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 101,378
  Trainable:    101,378
  ✓ Test passed!

EEGNet:
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 730
  Trainable:    730
  ✓ Test passed!

DE-CNN (VIGNet-style):
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 118,018
  Trainable:    118,018
  ✓ Test passed!

Channel Transformer:
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 70,562
  Trainable:    70,562
  ✓ Test passed!

All models tested successfully!


## Summary

This notebook defines:
- ✅ Data loading and preprocessing functions
- ✅ 4 PyTorch Dataset classes
- ✅ 6 model architectures (all tested with dummy data)

**Ready to import these into training notebooks!**